# Historical 2B benchmark evaluation

Historical-placement comparison retained for debugging and provenance. Do not combine it with the current core benchmark results.

In [ ]:
import json
from pathlib import Path

import pandas as pd

repo_root = Path.cwd()
if not (repo_root / "pyproject.toml").exists():
    repo_root = repo_root.parent

evaluation_root = repo_root / "outputs/evaluation"
runs = {
    "11325": "no_loc",
    "11326": "loc_text",
    "11327": "loc_embed",
}
shuffled_runs = {
    "11331": "loc_text",
    "11332": "loc_embed",
}
condition_order = ["no_loc", "loc_text", "loc_embed"]

def summary_path(job_id):
    return evaluation_root / job_id / "scored_predictions/summary.json"

all_jobs = runs | shuffled_runs
missing_jobs = [job_id for job_id in all_jobs if not summary_path(job_id).exists()]
if missing_jobs:
    raise FileNotFoundError(f"Sync these evaluation jobs first: {', '.join(missing_jobs)}")

summaries = {
    job_id: json.loads(summary_path(job_id).read_text(encoding="utf-8"))
    for job_id in all_jobs
}

task_type_rows = []
task_category_rows = []
caption_rows = []
for job_id, condition in runs.items():
    caption_rows.append({"condition": condition, "job_id": job_id, **summaries[job_id]["captioning"]})
    for row in summaries[job_id]["by_task_type"]:
        task_type_rows.append({"condition": condition, "job_id": job_id, **row})
    for row in summaries[job_id]["by_task_category"]:
        task_category_rows.append({"condition": condition, "job_id": job_id, **row})

task_type_scores = pd.DataFrame(task_type_rows)
task_category_scores = pd.DataFrame(task_category_rows)
caption_scores = pd.DataFrame(caption_rows)

## Main results

One primary metric per task family. Average rank gives each task family equal weight; lower is better.

In [ ]:
def task_score(condition, task_type, metric):
    row = task_type_scores[(task_type_scores["condition"] == condition) & (task_type_scores["task_type"] == task_type)]
    return row.iloc[0][metric]

main_results = pd.DataFrame(
    {
        "Caption BLEU-4": [caption_scores.loc[caption_scores["condition"] == c, "bleu4"].iloc[0] for c in condition_order],
        "Binary accuracy": [task_score(c, "binary", "accuracy") for c in condition_order],
        "MCQ accuracy": [task_score(c, "mcq", "accuracy") for c in condition_order],
        "Bounding box mIoU": [task_score(c, "bounding box", "miou") for c in condition_order],
    },
    index=condition_order,
)
score_columns = list(main_results.columns)
main_results["Average rank"] = main_results[score_columns].rank(ascending=False).mean(axis=1)
main_results.index.name = "Condition"
(
    main_results.style
    .format("{:.3f}")
    .highlight_max(subset=score_columns, props="font-weight: bold")
    .highlight_min(subset=["Average rank"], props="font-weight: bold")
    .set_caption("2B benchmark summary")
)

## Task-wise results

These tables use the main metric only. The full metric sets are listed at the bottom.

In [ ]:
def task_table(task_type, metric, overall_label="Overall"):
    categories = task_category_scores[task_category_scores["task_type"] == task_type]
    table = categories.pivot(index="condition", columns="task_category", values=metric)
    overall = task_type_scores[task_type_scores["task_type"] == task_type].set_index("condition")[metric]
    table.insert(0, overall_label, overall)
    table = table.reindex(condition_order)
    table.index.name = "Condition"
    table.columns.name = None
    return table

def show_task_table(table, caption):
    return table.style.format("{:.3f}").highlight_max(axis=0, props="font-weight: bold").set_caption(caption)

### Binary questions — accuracy

In [ ]:
binary_main = task_table("binary", "accuracy")
show_task_table(binary_main, "Binary accuracy by subtask")

### Multiple-choice questions — accuracy

In [ ]:
mcq_main = task_table("mcq", "accuracy")
show_task_table(mcq_main, "MCQ accuracy by subtask")

### Bounding boxes — mIoU

In [ ]:
bounding_box_main = task_table("bounding box", "miou")
show_task_table(bounding_box_main, "Bounding-box mIoU by subtask")

### Captioning — BLEU-4

In [ ]:
caption_main = caption_scores.set_index("condition").reindex(condition_order)[["bleu4"]]
caption_main.index.name = "Condition"
caption_main.columns = ["BLEU-4"]
show_task_table(caption_main, "Captioning BLEU-4")

## Shuffled-coordinate check

Compare each location-conditioned adapter with correct and shuffled coordinates. Negative deltas mean performance fell when the coordinates were shuffled.

In [ ]:
def primary_scores(summary):
    by_task = {row["task_type"]: row for row in summary["by_task_type"]}
    return {
        "Caption BLEU-4": summary["captioning"]["bleu4"],
        "Binary accuracy": by_task["binary"]["accuracy"],
        "MCQ accuracy": by_task["mcq"]["accuracy"],
        "Bounding box mIoU": by_task["bounding box"]["miou"],
    }

correct_jobs = {condition: job_id for job_id, condition in runs.items()}
shuffled_jobs = {condition: job_id for job_id, condition in shuffled_runs.items()}
coordinate_rows = []
for condition in ("loc_text", "loc_embed"):
    for coordinates, job_id in (("correct", correct_jobs[condition]), ("shuffled", shuffled_jobs[condition])):
        coordinate_rows.append(
            {"Condition": condition, "Coordinates": coordinates, "Job": job_id, **primary_scores(summaries[job_id])}
        )

coordinate_results = pd.DataFrame(coordinate_rows).set_index(["Condition", "Coordinates"])
coordinate_results.style.format({column: "{:.3f}" for column in score_columns}).set_caption("Correct versus shuffled coordinates")

In [ ]:
coordinate_deltas = pd.DataFrame(
    {
        condition: (
            coordinate_results.loc[(condition, "shuffled"), score_columns]
            - coordinate_results.loc[(condition, "correct"), score_columns]
        )
        for condition in ("loc_text", "loc_embed")
    }
).T
coordinate_deltas.index.name = "Condition"
coordinate_deltas.style.format("{:+.3f}").set_caption("Shuffled minus correct")

## Exhaustive diagnostic tables

The remaining tables retain sample counts, extraction rates, secondary metrics and job IDs for checking the results.

### Overall scores by task family (mixed task types)

In [ ]:
task_type_scores.sort_values(["task_type", "condition"]).reset_index(drop=True)

### Captioning — all metrics

In [ ]:
caption_scores.set_index("condition").reindex(condition_order).reset_index()

### Binary questions — all subtasks and diagnostics

In [ ]:
(
    task_category_scores[task_category_scores["task_type"] == "binary"]
    .sort_values(["task_category", "condition"])
    .reset_index(drop=True)
)

### Multiple-choice questions — all subtasks and diagnostics

In [ ]:
(
    task_category_scores[task_category_scores["task_type"] == "mcq"]
    .sort_values(["task_category", "condition"])
    .reset_index(drop=True)
)

### Bounding boxes — all subtasks and IoU thresholds

In [ ]:
(
    task_category_scores[task_category_scores["task_type"] == "bounding box"]
    .sort_values(["task_category", "condition"])
    .reset_index(drop=True)
)

### Shuffled-coordinate runs — all overall task metrics

In [ ]:
shuffled_task_rows = []
for job_id, condition in shuffled_runs.items():
    for row in summaries[job_id]["by_task_type"]:
        shuffled_task_rows.append({"condition": condition, "coordinates": "shuffled", "job_id": job_id, **row})
pd.DataFrame(shuffled_task_rows).sort_values(["task_type", "condition"]).reset_index(drop=True)

### Shuffled-coordinate runs — all captioning metrics

In [ ]:
pd.DataFrame(
    [
        {"condition": condition, "coordinates": "shuffled", "job_id": job_id, **summaries[job_id]["captioning"]}
        for job_id, condition in shuffled_runs.items()
    ]
).sort_values("condition").reset_index(drop=True)

### Shuffled-coordinate runs — all subtasks and diagnostics

In [ ]:
shuffled_category_rows = []
for job_id, condition in shuffled_runs.items():
    for row in summaries[job_id]["by_task_category"]:
        shuffled_category_rows.append({"condition": condition, "coordinates": "shuffled", "job_id": job_id, **row})
pd.DataFrame(shuffled_category_rows).sort_values(["task_type", "task_category", "condition"]).reset_index(drop=True)